## Dataset & Tokenizer

In [1]:
# path to dataset
DATA_PATH="../data/input.txt"

In [2]:
with open(DATA_PATH, "r") as file:
    data= file.read()

def find_unique_char(text: str) -> set[str]:
    return set(text)

def sort_char(text: set[str]) -> list[str]:
    return sorted(text)

In [3]:
text= find_unique_char(data)
sorted_text= sort_char(text)
vocab_size= len(sorted_text)

In [4]:
print(vocab_size)

65


In [5]:
stoi= {} # string to integer
for idx, char in enumerate(sorted_text):
    stoi[char]= idx

itos= {} # integer to string
for key, value in stoi.items():
    itos[value]= key

In [6]:
def encode(s: str) -> list[int]:
    encoded= [stoi[i] for i in s]
    return encoded

def decode(l: list[int]) -> str:
    decoded= "".join(itos[i] for i in l)
    return decoded

In [7]:
import torch

data_encoded= encode(data)
data_tensor= torch.tensor(data_encoded, dtype= torch.long)

In [8]:
n= int(0.9 * len(data_tensor))
train_data= data_tensor[:n]
test_data= data_tensor[n:]

In [9]:
EMBED_DIM= 32
BLOCK_SIZE= 8 # context length
BATCH_SIZE= 4
TRAIN_STEPS= 1000

### Batching - process multiple chunks simultaneously to take advantage of parallel processing

In [10]:
torch.manual_seed(7)

def get_batch(split, BATCH_SIZE, BLOCK_SIZE):
    data= train_data if split == "train" else val_data

    idx= torch.randint(len(data) - BLOCK_SIZE, (BATCH_SIZE, ))
    x= torch.stack([data[i: i + BLOCK_SIZE] for i in idx])
    y= torch.stack([data[i+1: i + BLOCK_SIZE + 1] for i in idx])

    return x, y

xb, yb = get_batch('train', BATCH_SIZE, BLOCK_SIZE)

In [11]:
print('inputs shape:', xb.shape)
print('targets shape:', yb.shape)
print('First row of x:', xb[0])
print('First row of y:', yb[0])

inputs shape: torch.Size([4, 8])
targets shape: torch.Size([4, 8])
First row of x: tensor([57,  1, 40, 59, 58,  1, 56, 47])
First row of y: tensor([ 1, 40, 59, 58,  1, 56, 47, 42])


You can notice that the x and y tensors are shifted by one character, which is a common technique in sequence modeling tasks. This allows the model to learn to predict the next character in a sequence based on the previous characters.

## Token Embeddings & Positional Encoding

In [12]:
import torch.nn as nn

token_embedding= nn.Embedding(vocab_size, EMBED_DIM)
posi_embedding= nn.Embedding(BLOCK_SIZE, EMBED_DIM)

token_emb= token_embedding(xb)

pos_emb= posi_embedding(torch.arange(BLOCK_SIZE))

x_emb= token_emb + pos_emb


In [13]:
print("Shape of tok_emb:", token_emb.shape)
print("Shape of x_emb:", x_emb.shape)

Shape of tok_emb: torch.Size([4, 8, 32])
Shape of x_emb: torch.Size([4, 8, 32])


You can read [4, 8, 32] as 4 independent sequences in batch, each containing 8 characters, and every character is now mathematically represented by a 32-dimensional continuous vector

## Self-Attention & Transformer Blocks

In [14]:
transformer_block= nn.TransformerEncoderLayer(d_model= EMBED_DIM, nhead= 4, dim_feedforward= 128, batch_first= True)

causal_mask= nn.Transformer.generate_square_subsequent_mask(BLOCK_SIZE)
hidden_state= transformer_block(x_emb, src_mask= causal_mask)

lm_head= nn.Linear(EMBED_DIM, vocab_size)
logits= lm_head(hidden_state)

print("Causal mask:\n", causal_mask)
print("Shape of hidden_states:", hidden_state.shape)
print("Shape of logits:", logits.shape)

Causal mask:
 tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
Shape of hidden_states: torch.Size([4, 8, 32])
Shape of logits: torch.Size([4, 8, 65])


The -inf values in the upper right of that matrix, turn into exact zeros when passed through a Softmax function. This mathematically blocks a character from "seeing" any characters that come after it, forcing the model to predict the future based only on the past (0.0 values).

In [15]:
class SmallLanguageModel(nn.Module):
    def __init__(self, EMBED_DIM, BLOCK_SIZE, vocab_size):
        super().__init__()
        self.token_embedding_table= nn.Embedding(vocab_size, EMBED_DIM)
        self.position_embedding_table= nn.Embedding(vocab_size, EMBED_DIM)

        self.blocks= nn.TransformerEncoderLayer(d_model= EMBED_DIM, nhead= 4, dim_feedforward= 128, batch_first= True)
        self.lm_head= nn.Linear(EMBED_DIM, vocab_size)

    def forward(self, idx):
        B, T= idx.shape
        token_emb= self.token_embedding_table(idx)
        pos_emb= self.position_embedding_table(torch.arange(T, device= idx.device))
        x= token_emb + pos_emb

        causal_mask= nn.Transformer.generate_square_subsequent_mask(T).to(idx.device)
        x= self.blocks(x, src_mask= causal_mask)


        logits= self.lm_head(x)
        return logits

model= SmallLanguageModel(EMBED_DIM, BLOCK_SIZE, vocab_size)
logits= model(xb)

print(logits.shape)

torch.Size([4, 8, 65])


In [16]:
import torch.nn.functional as F

optimizer= torch.optim.AdamW(model.parameters(), lr= 1e-3)

for step in range(TRAIN_STEPS):

    xb, yb= get_batch('train', BATCH_SIZE, BLOCK_SIZE)
    logits= model(xb)

    B, T, C= logits.shape
    logits_reshape= logits.view(B * T, C)
    targets_reshape= yb.view(B * T)

    loss= F.cross_entropy(logits_reshape, targets_reshape)

    optimizer.zero_grad(set_to_none= True)
    loss.backward()
    optimizer.step()

    if step % 100 == 0:
        print(f"Step {step}, Loss: {loss.item():.4f}")

print(f"Final Step - Loss: {loss.item():.4f}")

Step 0, Loss: 4.4463
Step 100, Loss: 3.8895
Step 200, Loss: 2.5766
Step 300, Loss: 3.2403
Step 400, Loss: 2.4950
Step 500, Loss: 2.8008
Step 600, Loss: 2.8514
Step 700, Loss: 3.2192
Step 800, Loss: 3.0224
Step 900, Loss: 2.6034
Final Step - Loss: 1.9930


In [17]:
def generate_text(model, idx, max_new_tokens):
    model.eval()

    for _ in range(max_new_tokens):

        idx_cond= idx[:, -BLOCK_SIZE:]
        with torch.no_grad():
            logits= model(idx_cond)

        logits= logits[:, -1, :]
        probab= F.softmax(logits, dim= -1)
        idx_next= torch.multinomial(probab, num_samples=1)
        idx= torch.cat((idx, idx_next), dim= 1)
    
    return idx

context= torch.zeros((1, 1), dtype= torch.long)
generated_idx= generate_text(model, context, max_new_tokens= 500)

generated_text= decode(generated_idx[0].tolist())

print(generated_text)


IB?m fareL:
lThy?
OMxoow?
Thacen'ea yas he ound thiser he bho.



SAUSEWagolot chenPeled ofraomiche hihsth Holrshen stheuBre thy worlo o csIy hiryersha  V
Scerise yod k EAENCJDCSOWR bave sothEhUbleror ile:
A ou, hathome yofwin th allafthad rll chpwn inK:

KENI noms,oHeCh aneerd choly roo h werlrane ;ee b butr urb, at CNanace thin me famo red isBr:
S arouor k r ve aome,
B sd sbUlsren sher tole thoves outhe me: nkyotree af fhy si,
in desif th ind bareler illime misat prd sh,the ye ber,, the thithe


You can see that although the model outputs mostly gibberish, it does learn capitalization, punctuation rules, and spaces. The reason for generating gibberish words is because of the strict constraints set for the CPU memory, which limits the model's ability to learn long-term dependencies in the text. To improve the quality, you increase the model's capacity and context window size.

Try changing the following parameters: <br>
BATCH_SIZE= 16<br>
BLOCK_SIZE= 32<br>
EMBED_DIM= 128<br>
TRAIN_STEPS= 2000<br>

After optimization, you will observe slight improvement in the quality of the generated text,and you might notice that the model mimics dialogue tags also. Even with a 32-charecter context, the model has only seen a fraction of the dataset during those 2000 steps, and because this is a charecter-level model using a <b><i>single</i></b> transformer block, it has to dictate its small neural capacity just to learn that "t-h-e" forms a word leaving little capacity for broader grammar rules.